In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Qiskit Runtimeローカルテストモード

<span id="test-locally"></span>


<details>
<summary><b>パッケージバージョン</b></summary>

このページのコードは、以下の要件を使用して開発されました。
これらのバージョン以降を使用することを推奨します。

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-aer~=0.17
```
</details>
ローカルテストモード（`qiskit-ibm-runtime` v0.22.0以降で利用可能）を使用すると、プログラムを微調整して実際の量子ハードウェアに送信する前にテストできます。ローカルテストモードでプログラムを検証した後は、QPUで実行するためにバックエンド名を変更するだけです。

ローカルテストモードを使用するには、Qiskit Runtimeプリミティブまたはセッションをインスタンス化する際に、``qiskit_ibm_runtime.fake_provider`` のフェイクバックエンドを指定するか、Qiskit Aerバックエンドを指定します。

- **フェイクバックエンド**: ``qiskit_ibm_runtime.fake_provider`` の[フェイクバックエンド](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider)は、QPUスナップショットを使用してIBM&reg; QPUの動作を模倣します。QPUスナップショットには、カップリングマップ、基底ゲート、量子ビット特性など、トランスパイラーのテストやQPUのノイズシミュレーションに有用なQPUに関する重要な情報が含まれています。スナップショットのノイズモデルはシミュレーション中に自動的に適用されます。
- **Aerシミュレーター**: [Qiskit Aer](/guides/simulate-with-qiskit-aer) のシミュレーターは、より大きな回路や[カスタムノイズモデル](/guides/build-noise-models)を扱える高性能なシミュレーションを提供します。ローカルテストモードで `AerSimulator` を使用すると、シミュレーション手法オプションの一覧が利用できます。多数の量子ビットを持つClifford回路を効率的にシミュレートする方法を示す[Cliffordシミュレーションモードの例](#clifford-sim)を参照してください。

    <details>
    <summary>**Qiskit Aerで利用可能なシミュレーション手法の一覧**</summary>

    詳細については、[`AerSimulator`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator) のドキュメントを参照してください。

    * ``"automatic"``: デフォルトのシミュレーション手法。回路とノイズモデルに基づいて、シミュレーション手法を自動的に選択します。

    * ``"statevector"``: 回路の末尾にすべての測定がある*理想的な*回路から測定結果をサンプリングできる密な状態ベクトルシミュレーション。ノイズシミュレーションの場合、各ショットはノイズモデルからランダムにサンプリングされたノイズ回路をサンプリングします。

    * ``"density_matrix"``: 回路の末尾にすべての測定がある*ノイズのある*回路から測定結果をサンプリングできる密度行列シミュレーション。

    * ``"stabilizer"``: ノイズモデル内のすべてのエラーもCliffordエラーである場合に、ノイズのあるClifford回路をシミュレートできる効率的なCliffordスタビライザー状態シミュレーター。

    * ``"extended_stabilizer"``: 状態をランク付きスタビライザー状態に分解することに基づく、Clifford + T回路の近似シミュレーター。項の数は非Clifford（T）ゲートの数とともに増加します。

    * ``"matrix_product_state"``: 状態にMatrix Product State（MPS）表現を使用するテンソルネットワーク状態ベクトルシミュレーター。シミュレーターオプションに応じて、MPSボンド次元を切り捨てるかどうかを選択できます。デフォルトの動作は切り捨てなしです。

    * ``"unitary"``: 理想的な回路の密なユニタリ行列シミュレーション。初期量子状態の発展ではなく、回路自体のユニタリ行列をシミュレートします。この手法はゲートのみをシミュレートし、測定、リセット、またはノイズをサポートしません。

    * ``"superop"``: 理想的またはノイズのある回路の密なsuperoperator行列シミュレーション。初期量子状態の発展ではなく、回路自体のsuperoperator行列をシミュレートします。この手法は理想的なゲートとノイズのあるゲートおよびリセットをシミュレートできますが、測定をサポートしません。

    * ``"tensor_network"``: 状態ベクトルと密度行列の両方をサポートするテンソルネットワークベースのシミュレーション。現在これはGPUのみで利用可能で、cuQuantumの `cuTensorNet` APIを使用して高速化されています。
    </details>

> **Note:** - ローカルテストモードではすべてのQiskit Runtimeオプションを指定できます。ただし、ローカルシミュレーターで実行した場合、ショット数以外のすべてのオプションは無視されます。
> - フェイクバックエンドまたはAerシミュレーターを使用する前に、`pip install qiskit-aer` を実行してQiskit Aerをインストールすることを推奨します。フェイクバックエンドは、パフォーマンスを活用するために、利用可能な場合はAerシミュレーターを内部で使用します。


## フェイクバックエンドの例

In [1]:
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using FakeManilaV2
fake_manila = FakeManilaV2()
pm = generate_preset_pass_manager(backend=fake_manila, optimization_level=1)
isa_qc = pm.run(qc)

# You can use a fixed seed to get fixed results.
options = {"simulator": {"seed_simulator": 42}}
sampler = Sampler(mode=fake_manila, options=options)

result = sampler.run([isa_qc]).result()

## AerSimulatorの例
セッションを使用した例（ノイズなし）：

In [2]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import Session, SamplerV2 as Sampler

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using AerSimulator.
# Session syntax is supported but ignored because local mode doesn't support sessions.
aer_sim = AerSimulator()
pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
with Session(backend=aer_sim) as session:
    sampler = Sampler(mode=session)
    result = sampler.run([isa_qc]).result()

ノイズありでシミュレートするには、QPU（量子ハードウェア）を指定してAerに送信します。Aerはそのタ QPUのキャリブレーションデータに基づいてノイズモデルを構築し、そのモデルを使用してAerバックエンドをインスタンス化します。必要に応じて、[ノイズモデルを構築する](/guides/build-noise-models)こともできます。

> **Caution:** QPUはさまざまな種類のノイズの影響を受ける可能性があります。ここで使用するQiskit Aerノイズモデルはそのうちの一部のみをシミュレートするため、実際のQPU上のノイズよりも軽度になる可能性があります。
> 
> QPUからノイズモデルを初期化する際に含まれるエラーの詳細については、Aerの [`NoiseModel`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.noise.NoiseModel.html#qiskit_aer.noise.NoiseModel.from_backend) APIリファレンスを参照してください。

ノイズありの例：

In [3]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler, QiskitRuntimeService

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

service = QiskitRuntimeService()

# Specify a QPU to use for the noise model
real_backend = service.backend("ibm_fez")
aer = AerSimulator.from_backend(real_backend)

# Run the sampler job locally using AerSimulator.
pm = generate_preset_pass_manager(backend=aer, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer)
result = sampler.run([isa_qc]).result()

<span id="clifford-sim"></span>
### Cliffordシミュレーション
Clifford回路は検証可能な結果で効率的にシミュレートできるため、Cliffordシミュレーションは非常に有用なツールです。詳細な例については、[Qiskit Aerプリミティブを使用したスタビライザー回路の効率的なシミュレーション](/guides/simulate-stabilizer-circuits)を参照してください。

例：

In [4]:
import numpy as np
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import SamplerV2 as Sampler

n_qubits = 500  # <---- note this uses 500 qubits!
circuit = efficient_su2(n_qubits)
circuit.measure_all()

rng = np.random.default_rng(1234)
params = rng.choice(
    [0, np.pi / 2, np.pi, 3 * np.pi / 2],
    size=circuit.num_parameters,
)

# Tell Aer to use the stabilizer (Clifford) simulation method
aer_sim = AerSimulator(method="stabilizer")

pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer_sim)
result = sampler.run([isa_qc]).result()

## 次のステップ
> **Tip:** - 詳細な[プリミティブの例](/guides/primitives-examples)を確認してください。
>     - [V2プリミティブへの移行](/guides/v2-primitives)を読んでください。
>     - IBM Quantum Learningの[コスト関数レッスン](/learning/courses/variational-algorithm-design/cost-functions)でプリミティブを実践してください。
>     - [トランスパイル](/guides/transpile)セクションでローカルでのトランスパイル方法を学んでください。
>     - [トランスパイラー設定の比較](/guides/circuit-transpilation-settings#compare-transpiler-settings)チュートリアルを試してください。